# 📘 Notebook 9: Common Probability Distributions in ML

**Week 2 — Calculus, Optimization & Probability Theory**

**Difficulty:** ⭐⭐⭐ (Intermediate)

---

## 🏠 Why Does This Matter?

Every ML model makes an **implicit assumption about the shape of the data distribution**.

When you use MSE loss for house price regression, you are assuming:
> "House prices follow a **Gaussian** distribution around the model's prediction"

When you classify houses as cheap/expensive with cross-entropy loss, you assume:
> "Each prediction follows a **Bernoulli** distribution"

When you count how many houses sell per day, you'd use:
> "Daily sales follow a **Poisson** distribution"

**Choosing the wrong distribution = choosing the wrong loss function = bad predictions.**

---

## 📚 Prerequisites
- Notebooks 6–8 (probability spaces, conditional probability, Bayes)

---

## Part 1: Discrete Distributions (counting outcomes)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import bernoulli, binom, poisson, norm, beta, dirichlet

# ─────────────────────────────────────────────────────────────────
# BERNOULLI DISTRIBUTION
#
# Models: a single yes/no outcome
# Parameter: p = probability of "yes"
# House price example: P(house is expensive) = p
#
# PMF: P(X=1) = p,  P(X=0) = 1-p
# Mean: p
# Variance: p(1-p)  [maximum uncertainty at p=0.5]
#
# ML use: binary classification output, the output of a sigmoid neuron
# ─────────────────────────────────────────────────────────────────

p_expensive = 0.3   # 30% of houses are expensive

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# ── Bernoulli ────────────────────────────────────────────────────
p_values = [0.2, 0.5, 0.8]
x_bern   = [0, 1]
width    = 0.2
for i, pv in enumerate(p_values):
    bars = axes[0].bar([x + i*width for x in x_bern], [1-pv, pv],
                        width=width, alpha=0.8, label=f'p={pv}')
axes[0].set_xticks([0.2, 1.2])
axes[0].set_xticklabels(['Not expensive (0)', 'Expensive (1)'], fontsize=10)
axes[0].set_title('Bernoulli: P(house is expensive)\n'
                  'Binary classification model output', fontsize=11)
axes[0].set_ylabel('Probability P(X=k)')
axes[0].legend(fontsize=9)
axes[0].set_ylim(0, 1.1)

# ── Binomial ─────────────────────────────────────────────────────
# P(exactly k houses are expensive out of N houses shown to a buyer)
n_houses_shown = 10   # buyer sees 10 houses
k_range = np.arange(0, n_houses_shown+1)
for pv, color in [(0.2, 'steelblue'), (0.5, 'green'), (0.8, 'orange')]:
    axes[1].plot(k_range, binom.pmf(k_range, n_houses_shown, pv),
                 'o-', linewidth=2, markersize=6, label=f'p={pv}')
axes[1].axvline(n_houses_shown*0.3, color='red', linestyle='--', linewidth=1.5,
                label='Expected count (p=0.3)')
axes[1].set_title('Binomial: How many of 10 houses are expensive?\n'
                  'Sum of independent Bernoulli outcomes', fontsize=11)
axes[1].set_xlabel('k (count of expensive houses)')
axes[1].set_ylabel('P(X=k)')
axes[1].legend(fontsize=9)

# ── Poisson ──────────────────────────────────────────────────────
# P(k houses sell in a day) given average λ sales/day
k_pois = np.arange(0, 20)
for lam, color in [(2, 'steelblue'), (5, 'green'), (10, 'orange')]:
    axes[2].plot(k_pois, poisson.pmf(k_pois, lam),
                 'o-', linewidth=2, markersize=5, label=f'λ={lam} sales/day')
axes[2].set_title('Poisson: How many houses sell per day?\n'
                  'Count data, average rate λ', fontsize=11)
axes[2].set_xlabel('k (houses sold today)')
axes[2].set_ylabel('P(X=k)')
axes[2].legend(fontsize=9)

plt.suptitle('Discrete Distributions — House Price Market Examples', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Part 2: Continuous Distributions (describing measurements)

### The Gaussian (Normal) Distribution — The Most Important in ML

### Plain English First

Many real-world quantities (heights, measurement errors, neural network weights after training) cluster around a central value with less probability further out — forming the famous **bell curve**.

For house prices, if we log-transform them (because prices are right-skewed), the log-prices follow roughly a Gaussian.

**Why Gaussian appears everywhere:**
- **Central Limit Theorem:** The sum of many independent random variables tends toward Gaussian, regardless of their original distribution
- **MSE loss** assumes Gaussian noise in your predictions
- **Weight initialization** in neural networks often uses Gaussian

$$f(x; \mu, \sigma) = \frac{1}{\sigma\sqrt{2\pi}} \exp\left(-\frac{(x-\mu)^2}{2\sigma^2}\right)$$

- `μ` (mu) = mean: where the center of the bell is
- `σ²` (sigma squared) = variance: how wide the bell is

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Gaussian Distribution: house price prediction uncertainty
#
# When your model predicts a house price of $350k,
# the Gaussian models the uncertainty around that prediction.
# σ = how uncertain you are.
# ─────────────────────────────────────────────────────────────────

x_prices = np.linspace(100_000, 700_000, 500)   # price range

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# ── Effect of mean (where the prediction is) ─────────────────────
sigma = 50_000   # fixed uncertainty: ±$50k
for mu, color, label in [
    (250_000, 'steelblue', '$250k prediction'),
    (350_000, 'green',     '$350k prediction'),
    (500_000, 'orange',    '$500k prediction'),
]:
    axes[0].plot(x_prices/1000, norm.pdf(x_prices, mu, sigma),
                 color=color, linewidth=2.5, label=label)
axes[0].set_title('Different predicted prices (μ), same uncertainty (σ)',
                  fontsize=11)
axes[0].set_xlabel('House Price ($thousands)'); axes[0].set_ylabel('Density')
axes[0].legend(fontsize=9); axes[0].grid(True, alpha=0.3)

# ── Effect of uncertainty (σ) ─────────────────────────────────────
mu = 350_000   # fixed prediction: $350k
for sigma, color, label in [
    (20_000, 'steelblue', 'σ=$20k (very confident)'),
    (50_000, 'green',     'σ=$50k (moderate)'),
    (100_000,'orange',    'σ=$100k (uncertain)'),
]:
    axes[1].plot(x_prices/1000, norm.pdf(x_prices, mu, sigma),
                 color=color, linewidth=2.5, label=label)
axes[1].set_title(f'Same prediction (μ=$350k), different uncertainty (σ)', fontsize=11)
axes[1].set_xlabel('House Price ($thousands)'); axes[1].set_ylabel('Density')
axes[1].legend(fontsize=9); axes[1].grid(True, alpha=0.3)

# ── 68-95-99.7 rule ───────────────────────────────────────────────
mu, sigma = 350_000, 50_000
x_fill = np.linspace(100_000, 700_000, 500)
y_fill = norm.pdf(x_fill, mu, sigma)
for n_sigma, color, pct in [(1,'blue','68%'), (2,'green','95%'), (3,'red','99.7%')]:
    mask = (x_fill >= mu-n_sigma*sigma) & (x_fill <= mu+n_sigma*sigma)
    axes[2].fill_between(x_fill[mask]/1000, y_fill[mask], alpha=0.3,
                         color=color, label=f'{pct} within ±{n_sigma}σ')
axes[2].plot(x_prices/1000, norm.pdf(x_prices, mu, sigma), 'k-', linewidth=2)
axes[2].set_title('68-95-99.7 Rule:\nHow to interpret uncertainty', fontsize=11)
axes[2].set_xlabel('House Price ($thousands)'); axes[2].set_ylabel('Density')
axes[2].legend(fontsize=9); axes[2].grid(True, alpha=0.3)

plt.suptitle('Gaussian Distribution: Modeling House Price Predictions + Uncertainty',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("68-95-99.7 Rule for a $350k prediction with σ=$50k:")
for n, pct in [(1,'68%'), (2,'95%'), (3,'99.7%')]:
    lo = (mu - n*sigma)/1000; hi = (mu + n*sigma)/1000
    print(f"  {pct} chance the true price is between ${lo:.0f}k and ${hi:.0f}k")

In [ ]:
# ─────────────────────────────────────────────────────────────────
# BETA DISTRIBUTION: uncertainty about a probability
#
# Used when the unknown quantity IS a probability.
# Example: "What fraction of houses in this neighborhood are expensive?"
#          We don't know the exact fraction — Beta models our uncertainty.
#
# Alpha = (prior expensive count + 1)
# Beta  = (prior cheap count + 1)
# ─────────────────────────────────────────────────────────────────

p_range = np.linspace(0.001, 0.999, 300)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: different Beta shapes
configs = [
    (1,  1,  'Uniform (no information yet)', 'gray'),
    (3,  10, 'Low fraction (~23%)',           'steelblue'),
    (10, 3,  'High fraction (~77%)',          'orange'),
    (20, 20, 'Concentrated near 50%',         'green'),
    (0.5, 0.5, 'Extreme: likely 0 or 1',      'red'),
]
for a, b, label, color in configs:
    axes[0].plot(p_range, beta.pdf(p_range, a, b), linewidth=2.5,
                 color=color, label=f'Beta({a},{b}): {label}')
axes[0].set_xlabel('p (fraction of expensive houses)')
axes[0].set_ylabel('Probability Density')
axes[0].set_title('Beta Distribution:\nUncertainty about a Probability', fontsize=11)
axes[0].legend(fontsize=8); axes[0].grid(True, alpha=0.3)

# Right: Bayesian updating with Beta distribution
# Start with Beta(1,1) (no info). Observe houses one by one.
# Each expensive house: α += 1. Each cheap house: β += 1.
alpha_prior, beta_prior = 1, 1
observations = [1, 1, 0, 1, 0, 0, 1, 1, 0, 1, 1, 0, 1, 0, 1]  # 1=expensive
checkpoints = [0, 3, 7, 15]

for n_obs in checkpoints:
    h = sum(observations[:n_obs])     # expensive houses seen so far
    t = n_obs - h                     # cheap houses seen so far
    a = alpha_prior + h               # posterior alpha
    b = beta_prior  + t               # posterior beta
    mean = a/(a+b)                    # expected fraction
    axes[1].plot(p_range, beta.pdf(p_range, a, b), linewidth=2.5,
                 label=f'After {n_obs} houses: {h} exp, {t} cheap → E[p]={mean:.2f}')

axes[1].set_xlabel('p (true fraction of expensive houses)')
axes[1].set_ylabel('Probability Density')
axes[1].set_title('Bayesian Updating with Beta Distribution:\nMore data → more certainty', fontsize=11)
axes[1].legend(fontsize=9); axes[1].grid(True, alpha=0.3)

plt.suptitle('Beta Distribution: Modeling Uncertainty About a Probability',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────────
# DIRICHLET DISTRIBUTION: the multi-class generalization of Beta
#
# Example: "What fraction of houses are cheap / medium / expensive?"
# The Dirichlet models uncertainty over this 3-way split.
#
# ML use: prior for the output probabilities of a classifier,
#         LDA topic models, Dirichlet process mixture models
# ─────────────────────────────────────────────────────────────────

np.random.seed(42)
K = 3   # three categories: cheap, medium, expensive

fig, axes = plt.subplots(2, 3, figsize=(16, 9))

configs_dir = [
    ([1, 1, 1],   'Uniform prior\nno preference'),
    ([0.5, 0.5, 0.5], 'Sparse prior\nprefers extreme predictions'),
    ([5, 5, 5],   'Concentrated prior\nprefers balanced predictions'),
    ([10, 2, 1],  'Biased: mostly cheap'),
    ([1, 10, 1],  'Biased: mostly medium'),
    ([1, 1, 10],  'Biased: mostly expensive'),
]

for ax, (alpha_vec, title) in zip(axes.flatten(), configs_dir):
    samples = np.random.dirichlet(alpha_vec, size=500)  # (500, 3) probability vectors

    # Project to 2D triangle (ternary plot)
    x_proj = samples[:,0] + 0.5 * samples[:,1]
    y_proj = samples[:,1] * np.sqrt(3)/2
    ax.scatter(x_proj, y_proj, alpha=0.3, s=10, color='steelblue')

    # Draw the triangle boundaries
    triangle = plt.Polygon([[0,0],[1,0],[0.5, np.sqrt(3)/2]],
                            fill=False, edgecolor='black', linewidth=2)
    ax.add_patch(triangle)

    # Label corners (100% of each category)
    offset = 0.05
    ax.text(-offset, -offset, 'P(cheap)=1', fontsize=8)
    ax.text(1+0.01, -offset,  'P(medium)=1', fontsize=8)
    ax.text(0.5-offset, np.sqrt(3)/2+0.02, 'P(exp)=1', fontsize=8)

    ax.set_xlim(-0.15, 1.15); ax.set_ylim(-0.1, 1.0)
    ax.axis('off')
    ax.set_title(f'Dir({alpha_vec})\n{title}', fontsize=10)

plt.suptitle('Dirichlet Distribution — Uncertainty Over 3-Class Probability Vectors\n'
             'Each point = one possible (P(cheap), P(medium), P(expensive)) vector',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("Dirichlet is used as the prior in LDA (Latent Dirichlet Allocation)")
print("In house pricing: prior over what fraction of listings fall in each price category")

## Part 3: Loss Functions Come from Distributions!

This is the key connection you need to internalize:

| Assumed data distribution | Loss function | ML task |
|--------------------------|--------------|----------|
| `y ∼ Gaussian(f(x;w), σ²)` | **MSE** `‖ŷ−y‖²` | Regression |
| `y ∼ Bernoulli(σ(f(x;w)))` | **Binary cross-entropy** | Binary classification |
| `y ∼ Categorical(softmax(f(x;w)))` | **Categorical cross-entropy** | Multi-class classification |
| `y ∼ Laplace(f(x;w), b)` | **MAE** `‖ŷ−y‖₁` | Robust regression |

**Each loss function IS a negative log-likelihood under a specific distribution assumption.**

Choosing your loss = choosing what distribution you think your data follows.

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Verify: MSE loss = negative log-likelihood of Gaussian
#
# If y_true ~ Gaussian(y_pred, σ²):
#   -log P(y_true | y_pred) = (y_true - y_pred)²/(2σ²) + constant
# Minimizing this = minimizing (y_true - y_pred)² = MSE!
# ─────────────────────────────────────────────────────────────────

np.random.seed(42)
N = 100
y_pred  = 350_000 * np.ones(N)    # model always predicts $350k
sigma   = 50_000                  # assumed noise level

# True prices scattered around $350k with noise
y_true = np.random.normal(350_000, sigma, N)

# MSE loss
mse = np.mean((y_pred - y_true)**2)

# Negative log-likelihood of Gaussian
nll_gaussian = np.mean(
    (y_pred - y_true)**2 / (2 * sigma**2)    # the quadratic term
    + np.log(sigma)                           # the normalization term
    + 0.5 * np.log(2 * np.pi)                # the constant
)

print("MSE loss vs Gaussian Negative Log-Likelihood")
print(f"  MSE                = {mse:.4f}")
print(f"  NLL (full)         = {nll_gaussian:.4f}")
print(f"  MSE = 2σ² × (NLL − constants) = {2*sigma**2*(nll_gaussian - np.log(sigma) - 0.5*np.log(2*np.pi)):.4f}")
print()
print("They're the same up to a scaling constant (2σ²) and additive constants.")
print("Minimizing MSE is equivalent to maximizing Gaussian likelihood!")
print()
print("Takeaway: When you use MSE loss for house price prediction, you're assuming:")
print("  'The actual price = my prediction + Gaussian noise with std =', sigma/1000, 'k dollars')")

---

## ✅ Self-Check Questions

**1.** Your model classifies houses as cheap/medium/expensive (3 classes).  
   Which distribution should the output probabilities follow?  
   Which distribution should you use as prior for the output if you want regularization?

**2.** The number of houses sold on a given day in a neighborhood averages 5.  
   You want to model P(exactly 8 houses sell tomorrow). Which distribution?  
   What are its parameters?

**3.** Your house price model uses MSE loss. A co-worker says: "Use MAE instead —  
   house prices have outliers (luxury mansions)." What distribution assumption does  
   MAE correspond to? Why is it better for outliers?

**4.** Beta(α=1, β=1) is the **uniform distribution** over [0,1].  
   You observe 8 expensive houses and 2 cheap houses. What is the posterior  
   distribution over p = P(expensive)?

**5.** Why does the Central Limit Theorem explain why Gaussian appears so often in ML?

<details>
<summary>Click to see answers</summary>

1. Output probabilities: **Categorical distribution** (3 classes). Prior for regularization: **Dirichlet distribution** (the conjugate prior for Categorical).

2. **Poisson distribution** with parameter λ=5 (average rate). P(X=8) = e⁻⁵ × 5⁸ / 8! ≈ 0.065.

3. MAE corresponds to a **Laplace distribution** (heavier tails than Gaussian). Better for outliers because the Laplace puts more probability on extreme values, so the loss function doesn't penalize outliers as severely (L1 vs L2).

4. Prior: Beta(1,1). After observing 8 expensive + 2 cheap: **posterior = Beta(1+8, 1+2) = Beta(9, 3)**. This has mean 9/(9+3) = 0.75 — reflecting our observation that 80% were expensive.

5. Neural network weights are the result of many small gradient updates, each affected by many independent training examples. The sum of many independent small effects → Gaussian (by CLT). This justifies Gaussian weight initialization and Gaussian priors on weights.

</details>

---

## 📌 Summary

| Distribution | Shape | House price use | ML loss |
|-------------|-------|----------------|----------|
| **Bernoulli(p)** | Two outcomes | Is house expensive? (yes/no) | Binary cross-entropy |
| **Binomial(n,p)** | Count successes in n trials | How many of 10 shown houses are expensive? | — |
| **Poisson(λ)** | Count events per unit time | How many houses sell per day? | — |
| **Gaussian(μ,σ²)** | Bell curve | Prediction with uncertainty | MSE |
| **Beta(α,β)** | Uncertainty over probability | What fraction are expensive? | Bayesian prior |
| **Dirichlet(α)** | Uncertainty over probability vector | Fraction in each price bracket | Prior for softmax |

**➡️ Next Notebook:** Expectation, Variance, Covariance — summarizing distributions with single numbers.